# Experiment 3: Centrifugal-Pump SRF Model

**Aim:** Study pressure rise, pump head, velocity components, and temperature rise in a simplified rotating annular passage.

**Numerical method:** Reduced-order polar single-rotating-frame model with prescribed radial/tangential velocities and steady thermal transport.

> **Scope:** This is not a blade-resolved pump CFD model. It is intended to expose centrifugal pressure rise, relative velocity, and loss heating.

This Google Colab notebook is self-contained. It reconstructs the tested solver inside the temporary Colab runtime, runs it, displays quantitative plots and an inline field contour, and packages every output file for download.

> **Colab workflow:** Choose a run mode, use **Runtime > Run all**, then download the results ZIP from the final cell. `quick` is only a workflow check; use `lab` or an approved `custom` case for submission.

In [ ]:
#@title 1. Prepare the self-contained Colab runtime

from pathlib import Path
import shutil
import subprocess
import sys
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None


def run_solver(script_path, arguments, output_dir, timeout=1800):
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(script_path), *[str(value) for value in arguments]]
    print("Running:", " ".join(command))
    completed = subprocess.run(command, cwd=output_dir, capture_output=True, text=True, timeout=timeout)
    (output_dir / "solver_stdout.log").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "solver_stderr.log").write_text(completed.stderr, encoding="utf-8")
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or f"Solver exited with code {completed.returncode}")
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return output_dir


def read_csv(path):
    data = np.genfromtxt(path, delimiter=",", names=True, dtype=None, encoding="utf-8")
    return np.atleast_1d(data)


def show_summary(path):
    text = Path(path).read_text(encoding="utf-8").strip()
    display(Markdown(f"### Solver summary\n```text\n{text}\n```"))
    return text


def list_outputs(output_dir):
    rows = []
    for path in sorted(Path(output_dir).iterdir()):
        if path.is_file():
            size = path.stat().st_size
            if size < 1024:
                size_text = f"{size} B"
            elif size < 1024**2:
                size_text = f"{size / 1024:.1f} KB"
            else:
                size_text = f"{size / 1024**2:.1f} MB"
            rows.append((path.name, size_text))
    table = "\n".join(f"| `{name}` | {size} |" for name, size in rows)
    display(Markdown(f"### Generated files\n| File | Size |\n|---|---:|\n{table}"))
    return rows


def read_structured_vtk(path):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    dims_index = next(i for i, line in enumerate(lines) if line.startswith("DIMENSIONS "))
    _, nx, ny, nz = lines[dims_index].split()
    nx, ny, nz = int(nx), int(ny), int(nz)
    if nz != 1:
        raise ValueError("Only 2D structured VTK files are supported by this notebook preview")
    points_index = next(i for i, line in enumerate(lines) if line.startswith("POINTS "))
    npoints = int(lines[points_index].split()[1])
    points = np.array([[float(v) for v in lines[points_index + 1 + k].split()[:3]] for k in range(npoints)])
    x = points[:, 0].reshape(ny, nx)
    y = points[:, 1].reshape(ny, nx)
    data_index = next(i for i, line in enumerate(lines) if line.startswith("POINT_DATA ")) + 1
    scalars = {}
    vectors = {}
    i = data_index
    while i < len(lines):
        if lines[i].startswith("SCALARS "):
            name = lines[i].split()[1]
            i += 2
            values = np.array([float(lines[i + k].split()[0]) for k in range(npoints)])
            scalars[name] = values.reshape(ny, nx)
            i += npoints
        elif lines[i].startswith("VECTORS "):
            name = lines[i].split()[1]
            i += 1
            values = np.array([[float(v) for v in lines[i + k].split()[:3]] for k in range(npoints)])
            vectors[name] = (values[:, 0].reshape(ny, nx), values[:, 1].reshape(ny, nx))
            i += npoints
        else:
            i += 1
    return x, y, scalars, vectors


def plot_vtk_scalar(path, preferred_field=None):
    path = Path(path)
    if not path.is_file():
        print(f"VTK preview skipped: {path.name} was not generated")
        return
    x, y, scalars, _ = read_structured_vtk(path)
    if not scalars:
        print("VTK preview skipped: no scalar field found")
        return
    field = preferred_field if preferred_field in scalars else next(iter(scalars))
    plt.figure(figsize=(8, 4.8))
    image = plt.contourf(x, y, scalars[field], levels=30)
    plt.colorbar(image, label=field.replace("_", " "))
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{field.replace('_', ' ').title()} - inline field preview")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()


def make_results_zip(output_dir, experiment_number):
    output_dir = Path(output_dir)
    archive = output_dir.parent / f"experiment_{experiment_number:02d}_results.zip"
    if archive.exists():
        archive.unlink()
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:
        for path in sorted(output_dir.rglob("*")):
            if path.is_file():
                handle.write(path, path.relative_to(output_dir))
    print(f"Results archive: {archive} ({archive.stat().st_size / 1024:.1f} KB)")
    return archive


EXPERIMENT = 3
ROOT = Path("/content/cfd_lab_exp03") if IN_COLAB else Path.cwd() / "_colab_runtime_exp03"
CODE_DIR = ROOT / "code"
OUTPUT_DIR = ROOT / "outputs"
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "common.py").write_text('from __future__ import annotations\n\nimport csv\nimport math\nfrom pathlib import Path\nfrom typing import Iterable, Mapping\n\nimport numpy as np\n\n\ndef trapezoidal_integral(values: np.ndarray, coordinates: np.ndarray) -> float:\n    y = np.asarray(values, dtype=float).ravel()\n    x = np.asarray(coordinates, dtype=float).ravel()\n    if y.size != x.size or y.size < 2:\n        raise ValueError("trapezoidal integration requires equally sized arrays with at least two points")\n    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))\n\n\ndef write_csv(path: str | Path, header: Iterable[str], rows: Iterable[Iterable[object]]) -> None:\n    with Path(path).open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(list(header))\n        writer.writerows(rows)\n\n\ndef mesh_xy(nx: int, ny: int, lx: float = 1.0, ly: float = 1.0, x0: float = 0.0, y0: float = 0.0) -> tuple[np.ndarray, np.ndarray]:\n    x = np.linspace(x0, x0 + lx, nx)\n    y = np.linspace(y0, y0 + ly, ny)\n    return np.meshgrid(x, y)\n\n\ndef write_structured_grid_vtk(\n    path: str | Path,\n    x: np.ndarray,\n    y: np.ndarray,\n    scalars: Mapping[str, np.ndarray] | None = None,\n    vectors: Mapping[str, tuple[np.ndarray, np.ndarray]] | None = None,\n    title: str = "CFD lab output",\n) -> None:\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y, dtype=float)\n    if x.shape != y.shape or x.ndim != 2:\n        raise ValueError("x and y must be equally shaped 2D arrays")\n    ny, nx = x.shape\n    scalar_map = dict(scalars or {})\n    vector_map = dict(vectors or {})\n    for name, values in scalar_map.items():\n        if np.asarray(values).shape != x.shape:\n            raise ValueError(f"scalar {name} has shape {np.asarray(values).shape}, expected {x.shape}")\n    for name, values in vector_map.items():\n        if len(values) != 2 or np.asarray(values[0]).shape != x.shape or np.asarray(values[1]).shape != x.shape:\n            raise ValueError(f"vector {name} has incompatible shape")\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET STRUCTURED_GRID\\n")\n        handle.write(f"DIMENSIONS {nx} {ny} 1\\n")\n        handle.write(f"POINTS {nx * ny} float\\n")\n        for j in range(ny):\n            for i in range(nx):\n                handle.write(f"{x[j, i]:.12e} {y[j, i]:.12e} 0\\n")\n        handle.write(f"POINT_DATA {nx * ny}\\n")\n        for name, values in scalar_map.items():\n            data = np.asarray(values, dtype=float)\n            handle.write(f"SCALARS {name} float 1\\n")\n            handle.write("LOOKUP_TABLE default\\n")\n            for value in data.ravel(order="C"):\n                handle.write(f"{value:.12e}\\n")\n        for name, values in vector_map.items():\n            vx = np.asarray(values[0], dtype=float)\n            vy = np.asarray(values[1], dtype=float)\n            handle.write(f"VECTORS {name} float\\n")\n            for ax, ay in zip(vx.ravel(order="C"), vy.ravel(order="C")):\n                handle.write(f"{ax:.12e} {ay:.12e} 0\\n")\n\n\ndef write_polyline_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, title: str = "polyline", close: bool = False) -> None:\n    x = np.asarray(x, dtype=float).ravel()\n    y = np.asarray(y, dtype=float).ravel()\n    if x.size != y.size or x.size < 2:\n        raise ValueError("polyline requires equally sized x and y arrays")\n    ids = list(range(x.size))\n    if close:\n        ids.append(0)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {x.size} float\\n")\n        for xi, yi in zip(x, y):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES 1 {len(ids) + 1}\\n")\n        handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_multiline_vtk(path: str | Path, lines: list[tuple[np.ndarray, np.ndarray]], title: str = "lines") -> None:\n    points: list[tuple[float, float]] = []\n    connectivity: list[list[int]] = []\n    for x, y in lines:\n        xa = np.asarray(x, dtype=float).ravel()\n        ya = np.asarray(y, dtype=float).ravel()\n        if xa.size != ya.size or xa.size < 2:\n            raise ValueError("each line needs equally sized x and y arrays")\n        ids = list(range(len(points), len(points) + xa.size))\n        points.extend(zip(xa.tolist(), ya.tolist()))\n        connectivity.append(ids)\n    size = sum(len(ids) + 1 for ids in connectivity)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {len(points)} float\\n")\n        for xi, yi in points:\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES {len(connectivity)} {size}\\n")\n        for ids in connectivity:\n            handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_points_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, scalars: Mapping[str, np.ndarray] | None = None, title: str = "points") -> None:\n    xa = np.asarray(x, dtype=float).ravel()\n    ya = np.asarray(y, dtype=float).ravel()\n    if xa.size != ya.size:\n        raise ValueError("point arrays must have equal size")\n    scalar_map = dict(scalars or {})\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {xa.size} float\\n")\n        for xi, yi in zip(xa, ya):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"VERTICES {xa.size} {2 * xa.size}\\n")\n        for i in range(xa.size):\n            handle.write(f"1 {i}\\n")\n        if scalar_map:\n            handle.write(f"POINT_DATA {xa.size}\\n")\n            for name, values in scalar_map.items():\n                data = np.asarray(values, dtype=float).ravel()\n                if data.size != xa.size:\n                    raise ValueError(f"scalar {name} has wrong size")\n                handle.write(f"SCALARS {name} float 1\\n")\n                handle.write("LOOKUP_TABLE default\\n")\n                for value in data:\n                    handle.write(f"{value:.12e}\\n")\n\n\ndef gradient(field: np.ndarray, dx: float, dy: float) -> tuple[np.ndarray, np.ndarray]:\n    gy, gx = np.gradient(field, dy, dx, edge_order=2)\n    return gx, gy\n\n\ndef laplacian(field: np.ndarray, dx: float, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    out[1:-1, 1:-1] = (\n        (field[1:-1, 2:] - 2.0 * field[1:-1, 1:-1] + field[1:-1, :-2]) / (dx * dx)\n        + (field[2:, 1:-1] - 2.0 * field[1:-1, 1:-1] + field[:-2, 1:-1]) / (dy * dy)\n    )\n    return out\n\n\ndef upwind_derivative_x(field: np.ndarray, velocity: np.ndarray, dx: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[1:-1, :-2]) / dx\n    forward = (field[1:-1, 2:] - field[1:-1, 1:-1]) / dx\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef upwind_derivative_y(field: np.ndarray, velocity: np.ndarray, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[:-2, 1:-1]) / dy\n    forward = (field[2:, 1:-1] - field[1:-1, 1:-1]) / dy\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef poisson_jacobi(phi: np.ndarray, rhs: np.ndarray, dx: float, dy: float, iterations: int, boundary: str = "neumann", mask: np.ndarray | None = None) -> np.ndarray:\n    dx2 = dx * dx\n    dy2 = dy * dy\n    denom = 2.0 * (dx2 + dy2)\n    for _ in range(iterations):\n        old = phi.copy()\n        phi[1:-1, 1:-1] = (\n            (old[1:-1, 2:] + old[1:-1, :-2]) * dy2\n            + (old[2:, 1:-1] + old[:-2, 1:-1]) * dx2\n            - rhs[1:-1, 1:-1] * dx2 * dy2\n        ) / denom\n        if mask is not None:\n            phi[mask > 0.5] = 0.0\n        if boundary == "neumann":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = 0.0\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n        elif boundary == "closed":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = phi[:, -2]\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n            phi -= np.mean(phi)\n        else:\n            raise ValueError("unsupported pressure boundary")\n    return phi\n\n\ndef tdma(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> np.ndarray:\n    a = np.asarray(a, dtype=float).copy()\n    b = np.asarray(b, dtype=float).copy()\n    c = np.asarray(c, dtype=float).copy()\n    d = np.asarray(d, dtype=float).copy()\n    n = d.size\n    if n == 0 or a.size != n or b.size != n or c.size != n:\n        raise ValueError("TDMA arrays must have equal nonzero size")\n    for i in range(1, n):\n        if abs(b[i - 1]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        factor = a[i] / b[i - 1]\n        b[i] -= factor * c[i - 1]\n        d[i] -= factor * d[i - 1]\n    x = np.empty(n, dtype=float)\n    if abs(b[-1]) < 1e-30:\n        raise FloatingPointError("zero TDMA pivot")\n    x[-1] = d[-1] / b[-1]\n    for i in range(n - 2, -1, -1):\n        if abs(b[i]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        x[i] = (d[i] - c[i] * x[i + 1]) / b[i]\n    return x\n\n\ndef relative_l2(numerical: np.ndarray, reference: np.ndarray) -> float:\n    a = np.asarray(numerical, dtype=float)\n    b = np.asarray(reference, dtype=float)\n    denominator = float(np.linalg.norm(b.ravel()))\n    numerator = float(np.linalg.norm((a - b).ravel()))\n    return numerator / max(denominator, 1e-30)\n\n\ndef require_finite(*arrays: np.ndarray) -> None:\n    for array in arrays:\n        if not np.all(np.isfinite(array)):\n            raise FloatingPointError("non-finite numerical field detected")\n\n\ndef positive_float(value: str, name: str) -> float:\n    result = float(value)\n    if not math.isfinite(result) or result <= 0.0:\n        raise ValueError(f"{name} must be positive and finite")\n    return result\n\n\ndef positive_int(value: str, name: str, minimum: int = 1) -> int:\n    result = int(value)\n    if result < minimum:\n        raise ValueError(f"{name} must be at least {minimum}")\n    return result\n', encoding="utf-8")
(CODE_DIR / 'exp03_centrifugal_pump_srf.py').write_text('from __future__ import annotations\n\nimport csv\nimport math\nimport sys\n\nimport numpy as np\n\nfrom common import require_finite, write_multiline_vtk, write_points_vtk, write_structured_grid_vtk\n\n\ndef main() -> int:\n    nr = int(sys.argv[1]) if len(sys.argv) > 1 else 80\n    nt = int(sys.argv[2]) if len(sys.argv) > 2 else 160\n    rpm = float(sys.argv[3]) if len(sys.argv) > 3 else 2000.0\n    reynolds = float(sys.argv[4]) if len(sys.argv) > 4 else 500.0\n    t_in = float(sys.argv[5]) if len(sys.argv) > 5 else 300.0\n    t_wall = float(sys.argv[6]) if len(sys.argv) > 6 else 330.0\n    iterations = int(sys.argv[7]) if len(sys.argv) > 7 else 1200\n    swirl = float(sys.argv[8]) if len(sys.argv) > 8 else 0.85\n    wall_htc = float(sys.argv[9]) if len(sys.argv) > 9 else 120.0\n    loss_fraction = float(sys.argv[10]) if len(sys.argv) > 10 else 0.04\n    if nr < 12 or nt < 24 or rpm <= 0.0 or reynolds <= 0.0 or iterations < 1 or wall_htc < 0.0 or loss_fraction < 0.0:\n        raise ValueError("usage: python exp03_centrifugal_pump_srf.py nr nt rpm Re Tin Twall iterations swirl wallHTC lossFraction")\n    r1 = 0.030\n    r2 = 0.120\n    depth = 0.010\n    rho = 997.0\n    mu = 0.0010\n    cp = 4180.0\n    conductivity = 0.60\n    gravity = 9.81\n    omega = rpm * 2.0 * math.pi / 60.0\n    hydraulic_diameter = 2.0 * depth\n    inlet_velocity = reynolds * mu / (rho * hydraulic_diameter)\n    r = np.linspace(r1, r2, nr)\n    theta = np.linspace(0.0, 2.0 * math.pi, nt, endpoint=False)\n    rr, tt = np.meshgrid(r, theta, indexing="ij")\n    x = rr * np.cos(tt)\n    y = rr * np.sin(tt)\n    dr = r[1] - r[0]\n    dtheta = theta[1] - theta[0]\n    u_r = inlet_velocity * r1 / rr\n    u_theta_abs = swirl * omega * rr\n    u_theta_rel = (swirl - 1.0) * omega * rr\n    u_abs_x = u_r * np.cos(tt) - u_theta_abs * np.sin(tt)\n    u_abs_y = u_r * np.sin(tt) + u_theta_abs * np.cos(tt)\n    u_rel_x = u_r * np.cos(tt) - u_theta_rel * np.sin(tt)\n    u_rel_y = u_r * np.sin(tt) + u_theta_rel * np.cos(tt)\n    pressure = 0.5 * rho * omega * omega * (rr * rr - r1 * r1) - 12.0 * mu * inlet_velocity * r1 * np.log(rr / r1) / (depth * depth)\n    alpha = conductivity / (rho * cp)\n    wall_lambda = wall_htc * 2.0 / (rho * cp * depth)\n    loss_heat = loss_fraction * omega * omega * rr * np.maximum(u_r, 0.0) / cp\n    temperature = np.full((nr, nt), t_in, dtype=float)\n    history: list[list[float]] = []\n    relax = 0.85\n    for iteration in range(1, iterations + 1):\n        old = temperature.copy()\n        west = np.vstack((np.full((1, nt), t_in), old[:-1, :]))\n        east = np.vstack((old[1:, :], old[-1:, :]))\n        south = np.roll(old, 1, axis=1)\n        north = np.roll(old, -1, axis=1)\n        ar = np.abs(u_r) / dr\n        at = np.abs(u_theta_rel) / np.maximum(rr * dtheta, 1e-30)\n        d_r = alpha / (dr * dr)\n        d_t = alpha / np.maximum(rr * rr * dtheta * dtheta, 1e-30)\n        angular_upwind = np.where(u_theta_rel >= 0.0, south, north)\n        candidate = (\n            ar * west\n            + at * angular_upwind\n            + d_r * (east + west)\n            + d_t * (north + south)\n            + wall_lambda * t_wall\n            + loss_heat\n        ) / np.maximum(ar + at + 2.0 * d_r + 2.0 * d_t + wall_lambda, 1e-30)\n        temperature = (1.0 - relax) * old + relax * candidate\n        temperature[0, :] = t_in\n        temperature[-1, :] = temperature[-2, :]\n        lower = min(t_in, t_wall) - 100.0\n        upper = max(t_in, t_wall) + 100.0\n        temperature = np.clip(temperature, lower, upper)\n        residual = float(np.max(np.abs(temperature - old)))\n        t_out = float(np.mean(temperature[-1, :]))\n        history.append([iteration, residual, float(np.min(temperature)), float(np.max(temperature)), t_out])\n        require_finite(temperature)\n        if iteration == 1 or iteration == iterations or iteration % max(1, iterations // 8) == 0:\n            print(f"iteration={iteration} residual={residual:.6e} outlet_temperature={t_out:.6f}")\n        if residual < 1e-10 and iteration > 50:\n            break\n    pressure_rise = float(np.mean(pressure[-1, :]) - np.mean(pressure[0, :]))\n    pump_head = pressure_rise / (rho * gravity)\n    outlet_temperature = float(np.mean(temperature[-1, :]))\n    inlet_area = 2.0 * math.pi * r1 * depth\n    mass_flow = rho * inlet_velocity * inlet_area\n    heat_gain = mass_flow * cp * (outlet_temperature - t_in)\n    speed_abs = np.sqrt(u_abs_x * u_abs_x + u_abs_y * u_abs_y)\n    speed_rel = np.sqrt(u_rel_x * u_rel_x + u_rel_y * u_rel_y)\n    write_structured_grid_vtk("pump_annulus_srf.vtk", x, y, {"temperature": temperature, "pressure": pressure, "absolute_speed": speed_abs, "relative_speed": speed_rel, "radius": rr}, {"absolute_velocity": (u_abs_x, u_abs_y), "relative_velocity": (u_rel_x, u_rel_y)}, "Centrifugal pump annular SRF model")\n    circle_theta = np.linspace(0.0, 2.0 * math.pi, 361)\n    inner_x = r1 * np.cos(circle_theta)\n    inner_y = r1 * np.sin(circle_theta)\n    outer_x = r2 * np.cos(circle_theta)\n    outer_y = r2 * np.sin(circle_theta)\n    write_multiline_vtk("pump_boundaries.vtk", [(inner_x, inner_y), (outer_x, outer_y)], "Pump annular boundaries")\n    guides: list[tuple[np.ndarray, np.ndarray]] = []\n    radial = np.linspace(r1, r2, 60)\n    for angle in np.linspace(0.0, 2.0 * math.pi, 13)[:-1]:\n        guides.append((radial * np.cos(angle), radial * np.sin(angle)))\n    write_multiline_vtk("pump_flow_path_guides.vtk", guides, "Pump radial flow guides")\n    blade_guides: list[tuple[np.ndarray, np.ndarray]] = []\n    for angle in np.linspace(0.0, 2.0 * math.pi, 9)[:-1]:\n        blade_angle = angle + 0.75 * (radial - r1) / (r2 - r1)\n        blade_guides.append((radial * np.cos(blade_angle), radial * np.sin(blade_angle)))\n    write_multiline_vtk("pump_blade_guides.vtk", blade_guides, "Pump blade guides")\n    seed_angles = np.linspace(0.0, 2.0 * math.pi, 24, endpoint=False)\n    write_points_vtk("pump_stream_seeds.vtk", 1.05 * r1 * np.cos(seed_angles), 1.05 * r1 * np.sin(seed_angles), title="Pump stream seeds")\n    with open("pump_history.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["iteration", "residual_linf", "minimum_temperature", "maximum_temperature", "outlet_temperature"])\n        writer.writerows(history)\n    with open("pump_summary.txt", "w", encoding="utf-8") as handle:\n        handle.write("Simplified 2D annular single-rotating-frame centrifugal pump heat-transfer model\\n")\n        handle.write(f"nr {nr}\\nnt {nt}\\nrpm {rpm:.12g}\\nomega_rad_per_s {omega:.12e}\\nRe {reynolds:.12g}\\n")\n        handle.write(f"r1_m {r1:.12e}\\nr2_m {r2:.12e}\\nchannel_depth_m {depth:.12e}\\n")\n        handle.write(f"Tin_K {t_in:.12e}\\nTwall_K {t_wall:.12e}\\nTout_K {outlet_temperature:.12e}\\n")\n        handle.write(f"pressure_rise_Pa {pressure_rise:.12e}\\npump_head_m {pump_head:.12e}\\n")\n        handle.write(f"inlet_velocity_m_per_s {inlet_velocity:.12e}\\nmass_flow_rate_kg_per_s {mass_flow:.12e}\\nfluid_heat_gain_W {heat_gain:.12e}\\n")\n        handle.write(f"final_residual_linf {history[-1][1]:.12e}\\n")\n        handle.write("model_scope teaching_level_reduced_order_srf\\n")\n    print("Wrote pump_annulus_srf.vtk, pump_boundaries.vtk, pump_flow_path_guides.vtk, pump_blade_guides.vtk, pump_stream_seeds.vtk, pump_history.csv, pump_summary.txt")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
SCRIPT = CODE_DIR / 'exp03_centrifugal_pump_srf.py'
print("Runtime:", "Google Colab" if IN_COLAB else "local notebook test")
print("Working folder:", ROOT)
print("No package installation or external file upload is required.")


## Parameters

Use `quick` to test the workflow, `lab` for the prescribed case, and `custom` only when changing the form values for the assignment.

In [ ]:
#@title 2. Select run mode and parameters
RUN_MODE = "quick" #@param ["quick", "lab", "custom"]

nr = 80 #@param {type:"integer"}
ntheta = 160 #@param {type:"integer"}
rpm = 2000.0 #@param {type:"number"}
Re = 500.0 #@param {type:"number"}
T_in = 300.0 #@param {type:"number"}
T_wall = 330.0 #@param {type:"number"}
iterations = 1200 #@param {type:"integer"}
swirl = 0.85 #@param {type:"number"}
wall_HTC = 120.0 #@param {type:"number"}
loss_fraction = 0.04 #@param {type:"number"}

QUICK_ARGS = [16, 24, 1500, 300, 300, 330, 8, 0.8, 100, 0.04]
LAB_ARGS = [80, 160, 2000, 500, 300, 330, 1200, 0.85, 120, 0.04]
CUSTOM_ARGS = [nr, ntheta, rpm, Re, T_in, T_wall, iterations, swirl, wall_HTC, loss_fraction]
ARGS = QUICK_ARGS if RUN_MODE == "quick" else LAB_ARGS if RUN_MODE == "lab" else CUSTOM_ARGS
print("Run mode:", RUN_MODE)
print("Arguments:", ARGS)


## Run the solver

In [ ]:
#@title 3. Run the numerical solver
OUT = run_solver(SCRIPT, ARGS, OUTPUT_DIR)


## Numerical summary

In [ ]:
#@title 4. Read the solver summary
summary_text = show_summary(OUT / 'pump_summary.txt')


## Quantitative plots

In [ ]:
#@title 5. Plot the primary numerical results
history = read_csv(OUT / "pump_history.csv")
plt.figure(figsize=(7, 4))
plt.semilogy(history["iteration"], history["residual_linf"])
plt.xlabel("Iteration")
plt.ylabel("L-infinity residual")
plt.title("Thermal convergence")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history["iteration"], history["outlet_temperature"])
plt.xlabel("Iteration")
plt.ylabel("Outlet temperature [K]")
plt.title("Outlet-temperature convergence")
plt.grid(True)
plt.show()

## Inline field visualization

This preview removes the need for ParaView during the basic laboratory workflow. The VTK file is still included in the downloaded results for optional advanced post-processing.

In [ ]:
#@title 6. Display the final field and generated files
plot_vtk_scalar(OUT / 'pump_annulus_srf.vtk', 'pressure')
files = list_outputs(OUT)


## Download results

In [ ]:
#@title 7. Package and download the results
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
RESULTS_ZIP = make_results_zip(OUT, 3)
if DOWNLOAD_RESULTS:
    if IN_COLAB:
        colab_files.download(str(RESULTS_ZIP))
    else:
        print("Local test mode. Archive retained at:", RESULTS_ZIP)
else:
    print("Set DOWNLOAD_RESULTS to True and run this cell when ready to download.")


## 7. Compulsory assignment

Run 1000, 2000, and 3000 rpm at fixed Reynolds number. Plot pump head against $RPM^2$ and test the expected near-linear scaling.

## Report checklist

- State the governing equations and boundary conditions.
- Report the complete parameter set and grid.
- Include residual or error history.
- Include the required quantitative comparison.
- Include at least one correctly labelled inline field figure; VTK/ParaView figures are optional unless specifically assigned.
- Distinguish observed numerical behaviour from physical interpretation.
- State the limitations of the supplied teaching model.